In [2]:
import bz2

def decompress_bz2(path_in, path_out):
    with bz2.open(path_in, 'rb') as f_in, open(path_out, 'wb') as f_out:
        f_out.write(f_in.read())

decompress_bz2(
    "/kaggle/input/datasets/bittlingmayer/amazonreviews/train.ft.txt.bz2",
    "/kaggle/working/train.ft.txt"
)

decompress_bz2(
    "/kaggle/input/datasets/bittlingmayer/amazonreviews/test.ft.txt.bz2",
    "/kaggle/working/test.ft.txt"
)


In [3]:
import numpy as np
import pandas as pd

data = []
with open("/kaggle/working/train.ft.txt", "r", encoding="utf-8") as f:
    for line in f:
        label, text = line.split(" ", 1)   # split only at FIRST space
        data.append([label, text.strip()])

df = pd.DataFrame(data, columns=["label", "text"])
df

,label,text
0,__label__2,Stuning even for the non-gamer: This sound tra...
1,__label__2,The best soundtrack ever to anything.: I'm rea...
2,__label__2,Amazing!: This soundtrack is my favorite music...
3,__label__2,Excellent Soundtrack: I truly like this soundt...
4,__label__2,"Remember, Pull Your Jaw Off The Floor After He..."
...,...,...
3599995,__label__1,Don't do it!!: The high chair looks great when...
3599996,__label__1,"Looks nice, low functionality: I have used thi..."
3599997,__label__1,"compact, but hard to clean: We have a small ho..."
3599998,__label__1,what is it saying?: not sure what this book is...


In [4]:
df["label"] = df["label"].map({"__label__1": 0, "__label__2": 1})
df

,label,text
0,1,Stuning even for the non-gamer: This sound tra...
1,1,The best soundtrack ever to anything.: I'm rea...
2,1,Amazing!: This soundtrack is my favorite music...
3,1,Excellent Soundtrack: I truly like this soundt...
4,1,"Remember, Pull Your Jaw Off The Floor After He..."
...,...,...
3599995,0,Don't do it!!: The high chair looks great when...
3599996,0,"Looks nice, low functionality: I have used thi..."
3599997,0,"compact, but hard to clean: We have a small ho..."
3599998,0,what is it saying?: not sure what this book is...


In [10]:
df_small = df.sample(1000, random_state=42).reset_index(drop=True)
df_small

,label,text
0,0,Expensive Junk: This product consists of a pie...
1,0,"Toast too dark: Even on the lowest setting, th..."
2,1,Excellent imagery...dumbed down story: I enjoy...
3,0,Are we pretending everyone is married?: The au...
4,0,Not worth your time: Might as well just use a ...
...,...,...
995,1,A Good Book!: I think this book is a all right...
996,1,best iron on the market: This iron cuts my iro...
997,1,The first professional packaging book I have: ...
998,1,the best choice i think: The same work conduct...


In [11]:
from sklearn.model_selection import train_test_split

train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df_small["text"].tolist(),
    df_small["label"].tolist(),
    test_size=0.3,        # 30% goes to val+test
    random_state=42,
    stratify=df_small["label"]
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,        # split 30% into 15% val + 15% test
    random_state=42,
    stratify=temp_labels
)


In [ ]:
val_texts,val_labels

In [18]:
def get_bert_embeddings(text_list, batch_size=8):
    features = []
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i : i + batch_size]
        
        # Tokenize
        inputs = tokenizer(batch, padding='max_length', truncation=True, max_length=128, return_tensors="np")
        
        # CAST TO INT32 to match the model's expectation
        bert_inputs = {
            'input_word_ids': tf.cast(inputs['input_ids'], tf.int32),
            'input_mask': tf.cast(inputs['attention_mask'], tf.int32),
            'input_type_ids': tf.cast(inputs['token_type_ids'], tf.int32)
        }
        
        # Extract
        outputs = bert_encoder(bert_inputs)
        features.append(outputs["pooled_output"].numpy())
        
    return np.vstack(features)

In [19]:
# You MUST run this before the model.fit() block
print("Processing Train texts... this might take a minute.")
X_train = get_bert_embeddings(train_texts)

print("Processing Validation texts...")
X_val = get_bert_embeddings(val_texts)

print("Processing Test texts...")
X_test = get_bert_embeddings(test_texts)

print("All features created! Now you can train the model.")

Processing Train texts... this might take a minute.
Processing Validation texts...
Processing Test texts...
All features created! Now you can train the model.


In [24]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

# 1. Build the simple classifier
model = Sequential([
    Input(shape=(128,)),           # BERT Small output is 128 dimensions
    Dense(32, activation='relu'),   # Hidden layer to learn patterns
    Dropout(0.2),                   # Prevents overfitting
    Dense(1, activation='sigmoid')  # Final output: 0 (Neg) or 1 (Pos)
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# 2. Train the model using the features you just created
print("Starting training...")
history = model.fit(
    X_train, np.array(train_labels),
    validation_data=(X_val, np.array(val_labels)),
    epochs=20,
    batch_size=32
)

# 3. Final Evaluation
print("\n--- Final Results ---")
loss, accuracy = model.evaluate(X_test, np.array(test_labels))
print(f"✅ Test Accuracy: {accuracy:.4f}")

loss, accuracy = model.evaluate(X_train, np.array(train_labels))
print(f"✅ Train Accuracy: {accuracy:.4f}")

Starting training...
Epoch 1/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.5104 - loss: 0.7291 - val_accuracy: 0.5333 - val_loss: 0.6805
Epoch 2/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5696 - loss: 0.6821 - val_accuracy: 0.5867 - val_loss: 0.6667
Epoch 3/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6030 - loss: 0.6560 - val_accuracy: 0.6133 - val_loss: 0.6714
Epoch 4/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6355 - loss: 0.6472 - val_accuracy: 0.5867 - val_loss: 0.6464
Epoch 5/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6152 - loss: 0.6463 - val_accuracy: 0.6400 - val_loss: 0.6460
Epoch 6/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6374 - loss: 0.6396 - val_accuracy: 0.6333 - val_loss: 0.6356
Epoch 7/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6376 - loss: 0.6274 - val_accuracy: 0.6467 - val_loss: 0.6325
Epoch 8/20
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6806 - loss: 0.6121 - val_accura